# Sentiment Analysis on IMDB Reviews using LSTM and Keras

<hr>

### Steps
<ol type="1">
    <li>Load the dataset (50K IMDB Movie Review)</li>
    <li>Clean Dataset</li>
    <li>Encode Sentiments</li>
    <li>Split Dataset</li>
    <li>Tokenize and Pad/Truncate Reviews</li>
    <li>Build Architecture/Model</li>
    <li>Train and Test</li>
</ol>

<hr>
<i>Import all the libraries needed</i>

In [33]:
import pandas as pd    # to load dataset
import numpy as np     # for mathematic equation
from nltk.corpus import stopwords   # to get collection of stopwords
from sklearn.model_selection import train_test_split       # for splitting dataset
from tensorflow.keras.preprocessing.text import Tokenizer  # to encode text to int
from tensorflow.keras.preprocessing.sequence import pad_sequences   # to do padding or truncating
from tensorflow.keras.models import Sequential     # the model
from tensorflow.keras.layers import Embedding, LSTM, Dense # layers of the architecture
from tensorflow.keras.callbacks import ModelCheckpoint   # save model
from tensorflow.keras.models import load_model   # load saved model
import re
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## My Internship Notes (Deep Learning Specialization — Course 5: Sequence Models)

**Goal:** Build an end-to-end sentiment classifier using an LSTM and understand the full NLP pipeline:
data → cleaning → tokenization → padding → Embedding → LSTM → Sigmoid.

**Key choices I applied:**
- Clean reviews by removing HTML tags and non-alphabet characters.
- Convert labels: negative → 0, positive → 1.
- Use Tokenizer + padding so all sequences have the same length for batching.
- Use Embedding to learn word representations.
- Use LSTM to capture sequence patterns.
- Use sigmoid output for binary classification.

**What I will show as proof:** dataset loading, preprocessing, tokenization shapes, model summary, training logs, evaluation metrics, and custom predictions.


<hr>
<i>Preview dataset</i>

In [34]:
data = pd.read_csv('IMDB Dataset.csv')

print(data)

                                                  review sentiment
0      One of the other reviewers has mentioned that ...  positive
1      A wonderful little production. <br /><br />The...  positive
2      I thought this was a wonderful way to spend ti...  positive
3      Basically there's a family where a little boy ...  negative
4      Petter Mattei's "Love in the Time of Money" is...  positive
...                                                  ...       ...
49995  I thought this movie did a down right good job...  positive
49996  Bad plot, bad dialogue, bad acting, idiotic di...  negative
49997  I am a Catholic taught in parochial elementary...  negative
49998  I'm going to have to disagree with the previou...  negative
49999  No one expects the Star Trek movies to be high...  negative

[50000 rows x 2 columns]


<hr>
<b>Stop Word</b> is a commonly used words in a sentence, usually a search engine is programmed to ignore this words (i.e. "the", "a", "an", "of", etc.)

<i>Declaring the english stop words</i>

In [35]:
english_stops = set(stopwords.words('english'))

<hr>

### Load and Clean Dataset

In the original dataset, the reviews are still dirty. There are still html tags, numbers, uppercase, and punctuations. This will not be good for training, so in <b>load_dataset()</b> function, beside loading the dataset using <b>pandas</b>, I also pre-process the reviews by removing html tags, non alphabet (punctuations and numbers), stop words, and lower case all of the reviews.

### Encode Sentiments
In the same function, I also encode the sentiments into integers (0 and 1). Where 0 is for negative sentiments and 1 is for positive sentiments.

In [36]:
def load_dataset():
    df = pd.read_csv('IMDB Dataset.csv')
    x_data = df['review']       # Reviews/Input
    y_data = df['sentiment']    # Sentiment/Output

    # PRE-PROCESS REVIEW
    x_data = x_data.replace({'<.*?>': ''}, regex = True)          # remove html tag
    x_data = x_data.replace({'[^A-Za-z]': ' '}, regex = True)     # remove non alphabet
    x_data = x_data.apply(lambda review: [w for w in review.split() if w not in english_stops])  # remove stop words
    x_data = x_data.apply(lambda review: [w.lower() for w in review])   # lower case
    
    # ENCODE SENTIMENT -> 0 & 1
    y_data = y_data.replace('positive', 1)
    y_data = y_data.replace('negative', 0)

    return x_data, y_data

x_data, y_data = load_dataset()

print('Reviews')
print(x_data, '\n')
print('Sentiment')
print(y_data)

/tmp/ipykernel_14508/1137108278.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_data = y_data.replace('negative', 0)


KeyboardInterrupt: 

<hr>

### Split Dataset
In this work, I decided to split the data into 80% of Training and 20% of Testing set using <b>train_test_split</b> method from Scikit-Learn. By using this method, it automatically shuffles the dataset. We need to shuffle the data because in the original dataset, the reviews and sentiments are in order, where they list positive reviews first and then negative reviews. By shuffling the data, it will be distributed equally in the model, so it will be more accurate for predictions.

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x_data, y_data, test_size = 0.2)

print('Train Set')
print(x_train, '\n')
print(x_test, '\n')
print('Test Set')
print(y_train, '\n')
print(y_test)

Train Set
39418    [i, know, everyone, said, movie, utter, crap, ...
1750     [i, must, say, i, fairly, disappointed, horror...
39879    [i, mistakenly, thought, art, film, bed, eats,...
11781    [sorry, everyone, i, know, supposed, art, film...
2896     [my, god, bad, who, impostor, pretending, ali,...
                               ...                        
41394    [for, read, rohinton, mistry, highly, respecte...
26658    [this, fan, made, short, film, pretends, previ...
47606    [you, know, robin, williams, god, bless, const...
9856     [john, ford, paid, wagons, tribute, special, p...
41845    [i, saw, first, came, video, little, sister, g...
Name: review, Length: 40000, dtype: object 

3336     [this, movie, changed, life, hogan, performanc...
46400    [this, movie, awesome, not, quite, good, leif,...
4518     [i, remember, seeing, another, murder, mystery...
25003    [the, movie, andaz, apna, apna, books, top, in...
46832    [i, understand, people, like, movie, people, f...
 

<hr>
<i>Function for getting the maximum review length, by calculating the mean of all the reviews length (using <b>numpy.mean</b>)</i>

In [ ]:
def get_max_length():
    review_length = []
    for review in x_train:
        review_length.append(len(review))

    return int(np.ceil(np.mean(review_length)))

<hr>

### Tokenize and Pad/Truncate Reviews
A Neural Network only accepts numeric data, so we need to encode the reviews. I use <b>tensorflow.keras.preprocessing.text.Tokenizer</b> to encode the reviews into integers, where each unique word is automatically indexed (using <b>fit_on_texts</b> method) based on <b>x_train</b>. <br>
<b>x_train</b> and <b>x_test</b> is converted into integers using <b>texts_to_sequences</b> method.

Each reviews has a different length, so we need to add padding (by adding 0) or truncating the words to the same length (in this case, it is the mean of all reviews length) using <b>tensorflow.keras.preprocessing.sequence.pad_sequences</b>.


<b>post</b>, pad or truncate the words in the back of a sentence<br>
<b>pre</b>, pad or truncate the words in front of a sentence

In [ ]:
# ENCODE REVIEW
token = Tokenizer(lower=False)    # no need lower, because already lowered the data in load_data()
token.fit_on_texts(x_train)
x_train = token.texts_to_sequences(x_train)
x_test = token.texts_to_sequences(x_test)

max_length = get_max_length()

x_train = pad_sequences(x_train, maxlen=max_length, padding='post', truncating='post')
x_test = pad_sequences(x_test, maxlen=max_length, padding='post', truncating='post')

total_words = len(token.word_index) + 1   # add 1 because of 0 padding

print('Encoded X Train\n', x_train, '\n')
print('Encoded X Test\n', x_test, '\n')
print('Maximum review length: ', max_length)

Encoded X Train
 [[    1    47   194 ...     0     0     0]
 [    1   111    58 ...     0     0     0]
 [    1 10490   102 ...    76    17     7]
 ...
 [   96    47  1696 ...   600  4591    49]
 [  211  1482  1481 ... 10836  3392   808]
 [    1   123    23 ...    32  1550  2815]] 

Encoded X Test
 [[   8    3 1098 ...    0    0    0]
 [   8    3 1100 ...    0    0    0]
 [   1  287  220 ...    0    0    0]
 ...
 [   1   31    3 ...    0    0    0]
 [  32 1506    8 ...    0    0    0]
 [   1   25 1732 ...    0    0    0]] 

Maximum review length:  130


<hr>

### Build Architecture/Model
<b>Embedding Layer</b>: in simple terms, it creates word vectors of each word in the <i>word_index</i> and group words that are related or have similar meaning by analyzing other words around them.

<b>LSTM Layer</b>: to make a decision to keep or throw away data by considering the current input, previous output, and previous memory. There are some important components in LSTM.
<ul>
    <li><b>Forget Gate</b>, decides information is to be kept or thrown away</li>
    <li><b>Input Gate</b>, updates cell state by passing previous output and current input into sigmoid activation function</li>
    <li><b>Cell State</b>, calculate new cell state, it is multiplied by forget vector (drop value if multiplied by a near 0), add it with the output from input gate to update the cell state value.</li>
    <li><b>Ouput Gate</b>, decides the next hidden state and used for predictions</li>
</ul>

<b>Dense Layer</b>: compute the input with the weight matrix and bias (optional), and using an activation function. I use <b>Sigmoid</b> activation function for this work because the output is only 0 or 1.

The optimizer is <b>Adam</b> and the loss function is <b>Binary Crossentropy</b> because again the output is only 0 and 1, which is a binary number.

In [44]:
# ARCHITECTURE
EMBED_DIM = 32
LSTM_OUT = 64

model = Sequential()
model.add(Embedding(total_words, EMBED_DIM, input_length = max_length))
model.add(LSTM(LSTM_OUT))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

model.summary()


/workspaces/sentiment-analysis-IMDB-Review-using-LSTM/.venv/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

<hr>

### Training
For training, it is simple. We only need to fit our <b>x_train</b> (input) and <b>y_train</b> (output/label) data. For this training, I use a mini-batch learning method with a <b>batch_size</b> of <i>128</i> and <i>5</i> <b>epochs</b>.

Also, I added a callback called **checkpoint** to save the model locally for every epoch if its accuracy improved from the previous epoch.

In [ ]:
checkpoint = ModelCheckpoint(
    'models/LSTM.h5',
    monitor='accuracy',
    save_best_only=True,
    verbose=1
)

In [ ]:
model.fit(x_train, y_train, batch_size = 128, epochs = 5, callbacks=[checkpoint])

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.6014 - loss: 0.6262
Epoch 1: accuracy improved from None to 0.72008, saving model to models/LSTM.h5



Epoch 1: finished saving model to models/LSTM.h5
313/313 ━━━━━━━━━━━━━━━━━━━━ 25s 74ms/step - accuracy: 0.7201 - loss: 0.5280
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.8761 - loss: 0.3651
Epoch 2: accuracy improved from 0.72008 to 0.88660, saving model to models/LSTM.h5



Epoch 2: finished saving model to models/LSTM.h5
313/313 ━━━━━━━━━━━━━━━━━━━━ 23s 75ms/step - accuracy: 0.8866 - loss: 0.3347
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - accuracy: 0.8648 - loss: 0.3786
Epoch 3: accuracy did not improve from 0.88660
313/313 ━━━━━━━━━━━━━━━━━━━━ 24s 76ms/step - accuracy: 0.7802 - loss: 0.4880
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - accuracy: 0.6380 - loss: 0.6229
Epoch 4: accuracy did not improve from 0.88660
313/313 ━━━━━━━━━━━━━━━━━━━━ 41s 75ms/step - accuracy: 0.6467 - loss: 0.6174
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 0.6528 - loss: 0.6162
Epoch 5: accuracy did not improve from 0.88660
313/313 ━━━━━━━━━━━━━━━━━━━━ 23s 72ms/step - accuracy: 0.6505 - loss: 0.6144


<hr>

### Testing
To evaluate the model, we need to predict the sentiment using our <b>x_test</b> data and comparing the predictions with <b>y_test</b> (expected output) data. Then, we calculate the accuracy of the model by dividing numbers of correct prediction with the total data. Resulted an accuracy of <b>86.63%</b>

In [ ]:
import numpy as np

# probabilities in [0,1]
y_prob = model.predict(x_test, batch_size=128).ravel()

# convert to 0/1 using threshold 0.5
y_pred = (y_prob >= 0.5).astype(int)

true = np.sum(y_pred == y_test)
print(f"Correct Prediction: {true}")
print(f"Wrong Prediction: {len(y_pred) - true}")
print(f"Accuracy: {true/len(y_pred)*100:.2f}")


79/79 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step
Correct Prediction: 6443
Wrong Prediction: 3557
Accuracy: 64.43


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(
    y_test, y_pred, target_names=["negative", "positive"]
))


Confusion Matrix:
 [[1779 3166]
 [ 391 4664]]

Classification Report:
               precision    recall  f1-score   support

    negative       0.82      0.36      0.50      4945
    positive       0.60      0.92      0.72      5055

    accuracy                           0.64     10000
   macro avg       0.71      0.64      0.61     10000
weighted avg       0.71      0.64      0.61     10000



---

### Load Saved Model

Load saved model and use it to predict a movie review statement's sentiment (positive or negative).

In [37]:
loaded_model = load_model('models/LSTM.h5')

Receives a review as an input to be predicted

In [38]:
review = str(input('Movie Review: '))

The input must be pre processed before it is passed to the model to be predicted

In [39]:
# Pre-process input
regex = re.compile(r'[^a-zA-Z\s]')
review = regex.sub('', review)
print('Cleaned: ', review)

words = review.split(' ')
filtered = [w for w in words if w not in english_stops]
filtered = ' '.join(filtered)
filtered = [filtered.lower()]

print('Filtered: ', filtered)

Cleaned:  
Filtered:  ['']


Once again, we need to tokenize and encode the words. I use the tokenizer which was previously declared because we want to encode the words based on words that are known by the model.

In [40]:
tokenize_words = token.texts_to_sequences(filtered)
tokenize_words = pad_sequences(tokenize_words, maxlen=max_length, padding='post', truncating='post')
print(tokenize_words)

[[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]]


This is the result of the prediction which shows the **confidence score** of the review statement.

In [41]:
result = loaded_model.predict(tokenize_words)
print(result)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step
[[0.9216998]]


In [42]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

def predict_sentiment(text, tokenizer, max_length):
    # Apply the SAME cleaning rules used in your preprocessing
    text = re.sub(r'<.*?>', ' ', text)        # remove HTML tags
    text = re.sub(r'[^A-Za-z\s]', ' ', text)  # keep letters only
    text = text.lower()
    text = " ".join(text.split())             # normalize spaces

    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=max_length, padding='post', truncating='post')

    prob = float(model.predict(pad)[0][0])
    label = "positive" if prob >= 0.5 else "negative"
    return label, prob

custom_reviews = [
    "The acting was brilliant and the story was amazing. I loved it!",
    "This movie was boring, too long, and a complete waste of time.",
    "It had some good moments, but overall it was just okay."
]

for r in custom_reviews:
    lbl, p = predict_sentiment(r, token, max_length)
    print(f"{lbl.upper()} ({p:.3f}) -> {r}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
POSITIVE (0.566) -> The acting was brilliant and the story was amazing. I loved it!
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
NEGATIVE (0.103) -> This movie was boring, too long, and a complete waste of time.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
POSITIVE (0.566) -> It had some good moments, but overall it was just okay.


If the confidence score is close to 0, then the statement is **negative**. On the other hand, if the confidence score is close to 1, then the statement is **positive**. I use a threshold of **0.7** to determine which confidence score is positive and negative, so if it is equal or greater than 0.7, it is **positive** and if it is less than 0.7, it is **negative**

In [43]:
if result >= 0.7:
    print('positive')
else:
    print('negative')

positive
